# dqt Detector Benchmark

Precision / Recall / F1 at default thresholds across three synthetic benchmarks:
1. **NAB-like time series** — spike, level-shift, contextual anomaly patterns
2. **Warehouse-shape tabular** — lognormal revenue, normal KPI with injected point outliers

All data is synthetic (no external downloads). Updated on every release.


In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

RNG = np.random.default_rng(42)
RESULTS = []  # list of dicts: {benchmark, detector, precision, recall, f1}


## 1. NAB-like time series benchmark

In [2]:
def nab_series(n=500, anomaly_frac=0.05, pattern="spike", rng=None):
    """Generate a labeled time series. Returns (values, labels)."""
    if rng is None: rng = np.random.default_rng(0)
    values = rng.normal(0, 1, n)
    labels = np.zeros(n, dtype=int)
    n_anom = int(n * anomaly_frac)
    anom_idx = np.arange(int(n * 0.8), int(n * 0.8) + n_anom)
    if pattern == "spike":
        values[anom_idx] += rng.choice([-1, 1], n_anom) * 8.0
    elif pattern == "level_shift":
        values[anom_idx] += 4.0
    elif pattern == "contextual":
        values[anom_idx] = rng.normal(0, 0.1, n_anom)  # too-quiet
    labels[anom_idx] = 1
    return values, labels

def pr_f1(labels, score, threshold):
    pred = int(score >= threshold)
    # Window-level: score applies to whole window
    if pred == 1:
        tp = int(labels.sum() > 0)
        fp = int(labels.sum() == 0)
    else:
        tp = 0
        fp = 0
    fn = 1 - tp
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return round(precision, 3), round(recall, 3), round(f1, 3)


In [3]:
from dqt.algorithms.timeseries.bocpd import BOCPDDetector

# Run BOCPD against all 3 patterns
for pattern in ["spike", "level_shift", "contextual"]:
    vals, labels = nab_series(500, 0.05, pattern, RNG)
    ref_df = pd.DataFrame({"v": vals[:300]})
    curr_df = pd.DataFrame({"v": vals[300:]})
    curr_labels = labels[300:]

    try:
        det = BOCPDDetector()
        state = det.fit(ref_df)
        result = det.score(curr_df, state)
        p, r, f1 = pr_f1(curr_labels, result.score, 0.50)
        RESULTS.append({"benchmark": f"nab_{pattern}", "detector": "bocpd",
                        "precision": p, "recall": r, "f1": f1})
        print(f"bocpd  nab_{pattern:<14}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
    except Exception as e:
        print(f"ERROR bocpd nab_{pattern}: {e}")
        RESULTS.append({"benchmark": f"nab_{pattern}", "detector": "bocpd",
                        "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


bocpd  nab_spike           P=1.000 R=1.000 F1=1.000  score=0.9911


bocpd  nab_level_shift     P=1.000 R=1.000 F1=1.000  score=0.7940
bocpd  nab_contextual      P=1.000 R=1.000 F1=1.000  score=0.8071


## 2. Warehouse-shape tabular benchmark

In [4]:
from dqt.algorithms.outliers_uni.mad import MADOutlierDetector

for shape, ref_gen, dirty_gen in [
    ("lognormal_revenue",
     lambda: pd.DataFrame({"v": RNG.lognormal(6, 0.5, 500)}),
     lambda: pd.DataFrame({"v": np.concatenate([RNG.lognormal(6, 0.5, 475), RNG.lognormal(9.5, 0.3, 25)])})),
    ("normal_kpi",
     lambda: pd.DataFrame({"v": RNG.normal(100, 10, 500)}),
     lambda: pd.DataFrame({"v": np.concatenate([RNG.normal(100, 10, 475), RNG.normal(160, 5, 25)])})),
]:
    ref = ref_gen()
    dirty = dirty_gen()
    has_outliers = np.array([0]*475 + [1]*25)  # last 25 rows are dirty

    try:
        det = MADOutlierDetector()
        state = det.fit(ref)
        result = det.score(dirty, state)
        # Score > 0.05 means the window is flagged as having outliers
        p, r, f1 = pr_f1(has_outliers, result.score, 0.05)
        RESULTS.append({"benchmark": f"warehouse_{shape}", "detector": "mad",
                        "precision": p, "recall": r, "f1": f1})
        print(f"mad    warehouse_{shape:<18}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
    except Exception as e:
        print(f"ERROR mad warehouse_{shape}: {e}")
        RESULTS.append({"benchmark": f"warehouse_{shape}", "detector": "mad",
                        "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


mad    warehouse_lognormal_revenue   P=1.000 R=1.000 F1=1.000  score=0.0500
mad    warehouse_normal_kpi          P=0.000 R=0.000 F1=0.000  score=0.0000


## 3. Basic detectors

These are aggregate detectors. We simulate the pre-computed aggregate rows that the runner would produce and pass them directly to `fit`/`score`.

**Benchmark**: inject 10 % nulls / violations into current; score > 0.05 means flagged.

In [ ]:

# ---------------------------------------------------------------------------
# Helpers: build aggregate-row DataFrames for BaseAggregateDetector subclasses.
# Each detector's fit()/score() receives a 1-row DataFrame with the column
# names that get_aggregations() would produce.
# ---------------------------------------------------------------------------

def _agg_df(**kwargs):
    return pd.DataFrame([kwargs])

# ---- NullFractionDetector -------------------------------------------------
try:
    from dqt.algorithms.basic.null_fraction import NullFractionDetector
    # clean ref: 0/500 nulls; dirty current: 50/500 nulls (10 %)
    ref_agg  = _agg_df(null_count=0,  total_count=500)
    curr_agg = _agg_df(null_count=50, total_count=500)
    det = NullFractionDetector()
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    # high null fraction (0.10) should flag (score > 0.05)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_null_10pct", "detector": "null_fraction", "precision": p, "recall": r, "f1": f1})
    print(f"null_fraction          P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR null_fraction: {e}")
    RESULTS.append({"benchmark": "basic_null_10pct", "detector": "null_fraction", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- CompletenessDetector -------------------------------------------------
try:
    from dqt.algorithms.basic.completeness import CompletenessDetector
    # ref: 100 % complete; dirty: 90 % complete — score (rate) = 0.90 < warn threshold
    ref_agg  = _agg_df(null_count=0,  total_count=500)
    curr_agg = _agg_df(null_count=50, total_count=500)
    det = CompletenessDetector()
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    # completeness rate 0.90; flag when rate < 0.95 (default warn); treat as pass here
    # for benchmark purposes a degraded completeness = anomaly present
    p, r, f1 = pr_f1(np.array([1]), 1.0 - result.score, 0.05)
    RESULTS.append({"benchmark": "basic_completeness_90pct", "detector": "completeness", "precision": p, "recall": r, "f1": f1})
    print(f"completeness           P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR completeness: {e}")
    RESULTS.append({"benchmark": "basic_completeness_90pct", "detector": "completeness", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- UniquenessDetector ---------------------------------------------------
try:
    from dqt.algorithms.basic.uniqueness import UniquenessDetector
    # ref: 100 % unique; dirty: 90 % unique (50 duplicates)
    ref_agg  = _agg_df(distinct_count=500, total_count=500)
    curr_agg = _agg_df(distinct_count=450, total_count=500)
    det = UniquenessDetector()
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), 1.0 - result.score, 0.05)
    RESULTS.append({"benchmark": "basic_uniqueness_90pct", "detector": "uniqueness", "precision": p, "recall": r, "f1": f1})
    print(f"uniqueness             P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR uniqueness: {e}")
    RESULTS.append({"benchmark": "basic_uniqueness_90pct", "detector": "uniqueness", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- VolumeDetector -------------------------------------------------------
try:
    from dqt.algorithms.basic.volume import VolumeDetector
    # ref: 1000 rows; dirty: 200 rows (80 % drop)
    ref_agg  = _agg_df(row_count=1000)
    curr_agg = _agg_df(row_count=200)
    det = VolumeDetector()
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.20)
    RESULTS.append({"benchmark": "basic_volume_drop80pct", "detector": "volume", "precision": p, "recall": r, "f1": f1})
    print(f"volume                 P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR volume: {e}")
    RESULTS.append({"benchmark": "basic_volume_drop80pct", "detector": "volume", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- RowCountInRangeDetector ----------------------------------------------
try:
    from dqt.algorithms.basic.volume import RowCountInRangeDetector
    # expect 800-1200 rows; current has only 200 (out of range)
    det = RowCountInRangeDetector(date_col="created_at", start_date="2024-01-01", end_date="2024-01-31",
                                  min_rows=800, max_rows=1200)
    ref_agg  = _agg_df(windowed_count=1000)
    curr_agg = _agg_df(windowed_count=200)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.5)
    RESULTS.append({"benchmark": "basic_row_count_in_range", "detector": "row_count_in_range", "precision": p, "recall": r, "f1": f1})
    print(f"row_count_in_range     P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR row_count_in_range: {e}")
    RESULTS.append({"benchmark": "basic_row_count_in_range", "detector": "row_count_in_range", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


In [ ]:

# ---- FreshnessDetector ----------------------------------------------------
try:
    from dqt.algorithms.basic.freshness import FreshnessDetector
    from datetime import datetime, timezone, timedelta
    # "latest_ts" is 2 hours ago → within warn (1 h) but below fail (24 h)
    # We directly pass a 1-row DataFrame with the latest_ts value
    two_hours_ago = datetime.now(timezone.utc) - timedelta(hours=2)
    ref_agg  = _agg_df(latest_ts=datetime.now(timezone.utc))
    curr_agg = _agg_df(latest_ts=two_hours_ago)
    det = FreshnessDetector(warn_seconds=3600, fail_seconds=86400)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    # 7200 s > warn threshold of 3600 → should flag
    p, r, f1 = pr_f1(np.array([1]), result.score, 3600)
    RESULTS.append({"benchmark": "basic_freshness_2h", "detector": "freshness_seconds_behind", "precision": p, "recall": r, "f1": f1})
    print(f"freshness_seconds_behind  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.1f}")
except Exception as e:
    print(f"ERROR freshness_seconds_behind: {e}")
    RESULTS.append({"benchmark": "basic_freshness_2h", "detector": "freshness_seconds_behind", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- ValueInRangeDetector -------------------------------------------------
try:
    from dqt.algorithms.basic.value_checks import ValueInRangeDetector
    # 10 % of values outside [0, 100]
    ref_agg  = _agg_df(violation_count=0,  total_count=500)
    curr_agg = _agg_df(violation_count=50, total_count=500)
    det = ValueInRangeDetector(min_val=0, max_val=100)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_value_in_range", "detector": "value_in_range", "precision": p, "recall": r, "f1": f1})
    print(f"value_in_range         P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR value_in_range: {e}")
    RESULTS.append({"benchmark": "basic_value_in_range", "detector": "value_in_range", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- SetMembershipDetector ------------------------------------------------
try:
    from dqt.algorithms.basic.value_checks import SetMembershipDetector
    ref_agg  = _agg_df(violation_count=0,  total_count=500)
    curr_agg = _agg_df(violation_count=50, total_count=500)
    det = SetMembershipDetector(allowed_values={"A", "B", "C"})
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_set_membership", "detector": "set_membership", "precision": p, "recall": r, "f1": f1})
    print(f"set_membership         P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR set_membership: {e}")
    RESULTS.append({"benchmark": "basic_set_membership", "detector": "set_membership", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- SetExclusionDetector -------------------------------------------------
try:
    from dqt.algorithms.basic.value_checks import SetExclusionDetector
    ref_agg  = _agg_df(violation_count=0,  total_count=500)
    curr_agg = _agg_df(violation_count=50, total_count=500)
    det = SetExclusionDetector(forbidden_values={"BAD", "INVALID"})
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_set_exclusion", "detector": "set_exclusion", "precision": p, "recall": r, "f1": f1})
    print(f"set_exclusion          P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR set_exclusion: {e}")
    RESULTS.append({"benchmark": "basic_set_exclusion", "detector": "set_exclusion", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- RegexMatchDetector ---------------------------------------------------
try:
    from dqt.algorithms.basic.value_checks import RegexMatchDetector
    ref_agg  = _agg_df(violation_count=0,  total_count=500)
    curr_agg = _agg_df(violation_count=50, total_count=500)
    det = RegexMatchDetector(pattern=r"^\d{4}-\d{2}-\d{2}$")
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_regex_match", "detector": "regex_match", "precision": p, "recall": r, "f1": f1})
    print(f"regex_match            P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR regex_match: {e}")
    RESULTS.append({"benchmark": "basic_regex_match", "detector": "regex_match", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- SqlAssertionDetector -------------------------------------------------
try:
    from dqt.algorithms.basic.sql_assertion import SqlAssertionDetector
    ref_agg  = _agg_df(violation_count=0,  total_count=500)
    curr_agg = _agg_df(violation_count=50, total_count=500)
    det = SqlAssertionDetector(condition="amount > 0")
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_sql_assertion", "detector": "sql_assertion_violation", "precision": p, "recall": r, "f1": f1})
    print(f"sql_assertion_violation P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR sql_assertion_violation: {e}")
    RESULTS.append({"benchmark": "basic_sql_assertion", "detector": "sql_assertion_violation", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- NumericMeanDetector --------------------------------------------------
try:
    from dqt.algorithms.basic.numeric import NumericMeanDetector
    # ref: mean=100, std=10; dirty: mean=130 (3σ shift)
    ref_agg  = _agg_df(mean=100.0, stddev=10.0)
    curr_agg = _agg_df(mean=130.0, stddev=10.0)
    det = NumericMeanDetector()
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 3.0)
    RESULTS.append({"benchmark": "basic_numeric_mean_3sigma", "detector": "numeric_mean", "precision": p, "recall": r, "f1": f1})
    print(f"numeric_mean           P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR numeric_mean: {e}")
    RESULTS.append({"benchmark": "basic_numeric_mean_3sigma", "detector": "numeric_mean", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- SumInRangeDetector ---------------------------------------------------
try:
    from dqt.algorithms.basic.numeric_bounds import SumInRangeDetector
    # sum out of range [10000, 20000]; current sum = 5000
    ref_agg  = _agg_df(agg_value=15000.0)
    curr_agg = _agg_df(agg_value=5000.0)
    det = SumInRangeDetector(min_val=10000, max_val=20000)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.5)
    RESULTS.append({"benchmark": "basic_sum_in_range", "detector": "sum_in_range", "precision": p, "recall": r, "f1": f1})
    print(f"sum_in_range           P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR sum_in_range: {e}")
    RESULTS.append({"benchmark": "basic_sum_in_range", "detector": "sum_in_range", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- MinInRangeDetector ---------------------------------------------------
try:
    from dqt.algorithms.basic.numeric_bounds import MinInRangeDetector
    ref_agg  = _agg_df(agg_value=5.0)
    curr_agg = _agg_df(agg_value=-50.0)  # min dropped below 0
    det = MinInRangeDetector(min_val=0.0, max_val=100.0)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.5)
    RESULTS.append({"benchmark": "basic_min_in_range", "detector": "min_in_range", "precision": p, "recall": r, "f1": f1})
    print(f"min_in_range           P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR min_in_range: {e}")
    RESULTS.append({"benchmark": "basic_min_in_range", "detector": "min_in_range", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- MaxInRangeDetector ---------------------------------------------------
try:
    from dqt.algorithms.basic.numeric_bounds import MaxInRangeDetector
    ref_agg  = _agg_df(agg_value=95.0)
    curr_agg = _agg_df(agg_value=9999.0)  # max spiked
    det = MaxInRangeDetector(min_val=0.0, max_val=200.0)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.5)
    RESULTS.append({"benchmark": "basic_max_in_range", "detector": "max_in_range", "precision": p, "recall": r, "f1": f1})
    print(f"max_in_range           P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR max_in_range: {e}")
    RESULTS.append({"benchmark": "basic_max_in_range", "detector": "max_in_range", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- MedianInRangeDetector ------------------------------------------------
try:
    from dqt.algorithms.basic.numeric_bounds import MedianInRangeDetector
    ref_agg  = _agg_df(agg_value=50.0)
    curr_agg = _agg_df(agg_value=300.0)  # median jumped
    det = MedianInRangeDetector(min_val=0.0, max_val=100.0)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.5)
    RESULTS.append({"benchmark": "basic_median_in_range", "detector": "median_in_range", "precision": p, "recall": r, "f1": f1})
    print(f"median_in_range        P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR median_in_range: {e}")
    RESULTS.append({"benchmark": "basic_median_in_range", "detector": "median_in_range", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- StdDevInRangeDetector ------------------------------------------------
try:
    from dqt.algorithms.basic.numeric_bounds import StdDevInRangeDetector
    ref_agg  = _agg_df(agg_value=10.0)
    curr_agg = _agg_df(agg_value=50.0)  # stddev exploded
    det = StdDevInRangeDetector(min_val=0.0, max_val=20.0)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.5)
    RESULTS.append({"benchmark": "basic_stddev_in_range", "detector": "stddev_in_range", "precision": p, "recall": r, "f1": f1})
    print(f"stddev_in_range        P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR stddev_in_range: {e}")
    RESULTS.append({"benchmark": "basic_stddev_in_range", "detector": "stddev_in_range", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- QuantileInRangeDetector ----------------------------------------------
try:
    from dqt.algorithms.basic.numeric_bounds import QuantileInRangeDetector
    ref_agg  = _agg_df(agg_value=90.0)
    curr_agg = _agg_df(agg_value=500.0)  # p95 way too high
    det = QuantileInRangeDetector(quantile=0.95, min_val=0.0, max_val=150.0)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.5)
    RESULTS.append({"benchmark": "basic_quantile_in_range", "detector": "quantile_in_range", "precision": p, "recall": r, "f1": f1})
    print(f"quantile_in_range      P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR quantile_in_range: {e}")
    RESULTS.append({"benchmark": "basic_quantile_in_range", "detector": "quantile_in_range", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- CardinalityInRangeDetector -------------------------------------------
try:
    from dqt.algorithms.basic.numeric_bounds import CardinalityInRangeDetector
    ref_agg  = _agg_df(agg_value=5.0)
    curr_agg = _agg_df(agg_value=500.0)  # cardinality exploded (PII leak?)
    det = CardinalityInRangeDetector(min_val=1, max_val=10)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.5)
    RESULTS.append({"benchmark": "basic_cardinality_in_range", "detector": "cardinality_in_range", "precision": p, "recall": r, "f1": f1})
    print(f"cardinality_in_range   P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR cardinality_in_range: {e}")
    RESULTS.append({"benchmark": "basic_cardinality_in_range", "detector": "cardinality_in_range", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


In [ ]:

# ---- MonotonicityDetector -------------------------------------------------
try:
    from dqt.algorithms.basic.monotonicity import MonotonicityDetector
    ref_df   = pd.DataFrame({"v": np.arange(100, dtype=float)})
    curr_df  = pd.DataFrame({"v": RNG.permutation(100).astype(float)})  # shuffled = not monotonic
    det = MonotonicityDetector(direction="increasing")
    state  = det.fit(ref_df)
    result = det.score(curr_df, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.5)
    RESULTS.append({"benchmark": "basic_monotonicity", "detector": "monotonicity", "precision": p, "recall": r, "f1": f1})
    print(f"monotonicity           P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR monotonicity: {e}")
    RESULTS.append({"benchmark": "basic_monotonicity", "detector": "monotonicity", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- ValidityDetector -----------------------------------------------------
try:
    from dqt.algorithms.basic.validity import ValidityDetector
    ref_agg  = _agg_df(invalid_count=0,  total_count=500)
    curr_agg = _agg_df(invalid_count=50, total_count=500)  # 10 % invalid
    det = ValidityDetector(sql_predicate="amount > 0")
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), 1.0 - result.score, 0.05)
    RESULTS.append({"benchmark": "basic_validity", "detector": "validity", "precision": p, "recall": r, "f1": f1})
    print(f"validity               P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR validity: {e}")
    RESULTS.append({"benchmark": "basic_validity", "detector": "validity", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- StringLengthRangeDetector --------------------------------------------
try:
    from dqt.algorithms.basic.value_checks import StringLengthRangeDetector
    ref_agg  = _agg_df(violation_count=0,  total_count=500)
    curr_agg = _agg_df(violation_count=50, total_count=500)
    det = StringLengthRangeDetector(min_len=3, max_len=50)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_string_length_range", "detector": "string_length_range", "precision": p, "recall": r, "f1": f1})
    print(f"string_length_range    P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR string_length_range: {e}")
    RESULTS.append({"benchmark": "basic_string_length_range", "detector": "string_length_range", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- StringCaseDetector ---------------------------------------------------
try:
    from dqt.algorithms.basic.string_case import StringCaseDetector
    ref_agg  = _agg_df(violation_count=0,  total_count=500)
    curr_agg = _agg_df(violation_count=50, total_count=500)
    det = StringCaseDetector(case="upper")
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_string_case_violation", "detector": "string_case_violation", "precision": p, "recall": r, "f1": f1})
    print(f"string_case_violation  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR string_case_violation: {e}")
    RESULTS.append({"benchmark": "basic_string_case_violation", "detector": "string_case_violation", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- DateFormatDetector ---------------------------------------------------
try:
    from dqt.algorithms.basic.value_checks import DateFormatDetector
    ref_agg  = _agg_df(violation_count=0,  total_count=500)
    curr_agg = _agg_df(violation_count=50, total_count=500)
    det = DateFormatDetector(date_format="%Y-%m-%d")
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_date_format", "detector": "date_format", "precision": p, "recall": r, "f1": f1})
    print(f"date_format            P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR date_format: {e}")
    RESULTS.append({"benchmark": "basic_date_format", "detector": "date_format", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- DatePartCompletenessDetector -----------------------------------------
try:
    from dqt.algorithms.basic.date_part import DatePartCompletenessDetector
    # 10/30 daily buckets missing
    ref_agg  = _agg_df(missing_buckets=0,  total_buckets=30)
    curr_agg = _agg_df(missing_buckets=10, total_buckets=30)
    det = DatePartCompletenessDetector(col="created_at", granularity="day", lookback_days=30)
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_date_part_missing", "detector": "date_part_missing_fraction", "precision": p, "recall": r, "f1": f1})
    print(f"date_part_missing_fraction P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR date_part_missing_fraction: {e}")
    RESULTS.append({"benchmark": "basic_date_part_missing", "detector": "date_part_missing_fraction", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- ColumnPairComparisonDetector -----------------------------------------
try:
    from dqt.algorithms.basic.column_pairs import ColumnPairComparisonDetector
    ref_agg  = _agg_df(violation_count=0,  total_count=500)
    curr_agg = _agg_df(violation_count=50, total_count=500)
    det = ColumnPairComparisonDetector(col_a="end_date", col_b="start_date", operator=">")
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_column_pair_comparison", "detector": "column_pair_comparison", "precision": p, "recall": r, "f1": f1})
    print(f"column_pair_comparison P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR column_pair_comparison: {e}")
    RESULTS.append({"benchmark": "basic_column_pair_comparison", "detector": "column_pair_comparison", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- CompositeUniquenessDetector ------------------------------------------
try:
    from dqt.algorithms.basic.column_pairs import CompositeUniquenessDetector
    # 10 % duplicate composite keys
    ref_agg  = _agg_df(total_count=500, distinct_count=500)
    curr_agg = _agg_df(total_count=500, distinct_count=450)
    det = CompositeUniquenessDetector(key_columns=["user_id", "event_date"])
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.05)
    RESULTS.append({"benchmark": "basic_composite_uniqueness", "detector": "composite_uniqueness", "precision": p, "recall": r, "f1": f1})
    print(f"composite_uniqueness   P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR composite_uniqueness: {e}")
    RESULTS.append({"benchmark": "basic_composite_uniqueness", "detector": "composite_uniqueness", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


## 4. Drift detectors

**Benchmark**: ref = Normal(0, 1) 500 rows; current = Normal(2, 1) 100 rows (2σ shift). A high score means drift detected.

In [ ]:

_drift_ref  = pd.DataFrame({"v": RNG.normal(0, 1, 500)})
_drift_curr = pd.DataFrame({"v": RNG.normal(2, 1, 100)})  # 2σ shift
_drift_labels = np.array([1])  # drift present

def _drift_result(slug, det, ref, curr, threshold):
    state  = det.fit(ref)
    result = det.score(curr, state)
    p, r, f1 = pr_f1(_drift_labels, result.score, threshold)
    RESULTS.append({"benchmark": "drift_normal_2sigma", "detector": slug, "precision": p, "recall": r, "f1": f1})
    print(f"{slug:<30}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")

# ---- PSIDetector ----------------------------------------------------------
try:
    from dqt.algorithms.drift.psi import PSIDetector
    _drift_result("psi", PSIDetector(), _drift_ref, _drift_curr, threshold=0.10)
except Exception as e:
    print(f"ERROR psi: {e}")
    RESULTS.append({"benchmark": "drift_normal_2sigma", "detector": "psi", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- KS2SampleDetector ----------------------------------------------------
try:
    from dqt.algorithms.drift.ks2sample import KS2SampleDetector
    _drift_result("ks_pvalue", KS2SampleDetector(), _drift_ref, _drift_curr, threshold=0.95)
except Exception as e:
    print(f"ERROR ks_pvalue: {e}")
    RESULTS.append({"benchmark": "drift_normal_2sigma", "detector": "ks_pvalue", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- JSDivergenceDetector -------------------------------------------------
try:
    from dqt.algorithms.drift.divergence import JSDivergenceDetector
    _drift_result("js_divergence", JSDivergenceDetector(), _drift_ref, _drift_curr, threshold=0.10)
except Exception as e:
    print(f"ERROR js_divergence: {e}")
    RESULTS.append({"benchmark": "drift_normal_2sigma", "detector": "js_divergence", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- KLDivergenceDetector -------------------------------------------------
try:
    from dqt.algorithms.drift.divergence import KLDivergenceDetector
    _drift_result("kl_divergence", KLDivergenceDetector(), _drift_ref, _drift_curr, threshold=0.10)
except Exception as e:
    print(f"ERROR kl_divergence: {e}")
    RESULTS.append({"benchmark": "drift_normal_2sigma", "detector": "kl_divergence", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- Wasserstein1Detector -------------------------------------------------
try:
    from dqt.algorithms.drift.wasserstein import Wasserstein1Detector
    _drift_result("wasserstein_1", Wasserstein1Detector(), _drift_ref, _drift_curr, threshold=0.20)
except Exception as e:
    print(f"ERROR wasserstein_1: {e}")
    RESULTS.append({"benchmark": "drift_normal_2sigma", "detector": "wasserstein_1", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- MMDDetector ----------------------------------------------------------
try:
    from dqt.algorithms.drift.mmd import MMDDetector
    _drift_result("mmd", MMDDetector(), _drift_ref, _drift_curr, threshold=0.10)
except Exception as e:
    print(f"ERROR mmd: {e}")
    RESULTS.append({"benchmark": "drift_normal_2sigma", "detector": "mmd", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- ChiSquareDriftDetector (categorical) ---------------------------------
try:
    from dqt.algorithms.drift.chi_square import ChiSquareDriftDetector
    # ref: balanced A/B/C; current: heavily skewed to A
    _cat_ref  = pd.DataFrame({"v": RNG.choice(["A", "B", "C"], 500)})
    _cat_curr = pd.DataFrame({"v": RNG.choice(["A", "B", "C"], 100, p=[0.95, 0.025, 0.025])})
    det = ChiSquareDriftDetector()
    state  = det.fit(_cat_ref)
    result = det.score(_cat_curr, state)
    p, r, f1 = pr_f1(_drift_labels, result.score, 0.95)
    RESULTS.append({"benchmark": "drift_categorical_skewed", "detector": "chi_square_drift", "precision": p, "recall": r, "f1": f1})
    print(f"{'chi_square_drift':<30}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR chi_square_drift: {e}")
    RESULTS.append({"benchmark": "drift_categorical_skewed", "detector": "chi_square_drift", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- ADWINDetector --------------------------------------------------------
try:
    from dqt.algorithms.drift.adwin import ADWINDetector
    _drift_result("adwin", ADWINDetector(), _drift_ref, _drift_curr, threshold=0.5)
except Exception as e:
    print(f"ERROR adwin: {e}")
    RESULTS.append({"benchmark": "drift_normal_2sigma", "detector": "adwin", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


## 5. Univariate outlier detectors (remaining)

**Benchmark**: ref = Normal(0, 1) 500 rows; current = 500 rows with 5 % injected at ±8σ. Score > threshold means outliers detected.

In [ ]:

_uni_ref_arr  = RNG.normal(0, 1, 500)
_uni_curr_arr = np.concatenate([RNG.normal(0, 1, 475),
                                RNG.choice([-1, 1], 25) * 8.0])  # 5 % at ±8σ
_uni_ref  = pd.DataFrame({"v": _uni_ref_arr})
_uni_curr = pd.DataFrame({"v": _uni_curr_arr})
_uni_labels = np.array([1])  # outliers present

def _uni_result(slug, det, threshold=0.01):
    state  = det.fit(_uni_ref)
    result = det.score(_uni_curr, state)
    p, r, f1 = pr_f1(_uni_labels, result.score, threshold)
    RESULTS.append({"benchmark": "outliers_uni_5pct_8sigma", "detector": slug, "precision": p, "recall": r, "f1": f1})
    print(f"{slug:<35}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")

# ---- ZScoreDetector -------------------------------------------------------
try:
    from dqt.algorithms.outliers_uni.zscore import ZScoreDetector
    _uni_result("zscore_outlier_fraction", ZScoreDetector(threshold=3.0))
except Exception as e:
    print(f"ERROR zscore_outlier_fraction: {e}")
    RESULTS.append({"benchmark": "outliers_uni_5pct_8sigma", "detector": "zscore_outlier_fraction", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- IQRFenceDetector -----------------------------------------------------
try:
    from dqt.algorithms.outliers_uni.iqr_fence import IQRFenceDetector
    _uni_result("iqr_fence", IQRFenceDetector(k=3.0))
except Exception as e:
    print(f"ERROR iqr_fence: {e}")
    RESULTS.append({"benchmark": "outliers_uni_5pct_8sigma", "detector": "iqr_fence", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- AdjustedBoxplotDetector ----------------------------------------------
try:
    from dqt.algorithms.outliers_uni.adjusted_boxplot import AdjustedBoxplotDetector
    _uni_result("adjusted_boxplot_fraction", AdjustedBoxplotDetector(h=2.5))
except Exception as e:
    print(f"ERROR adjusted_boxplot_fraction: {e}")
    RESULTS.append({"benchmark": "outliers_uni_5pct_8sigma", "detector": "adjusted_boxplot_fraction", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- GrubbsDetector -------------------------------------------------------
try:
    from dqt.algorithms.outliers_uni.grubbs import GrubbsDetector
    _uni_result("grubbs", GrubbsDetector(), threshold=0.95)
except Exception as e:
    print(f"ERROR grubbs: {e}")
    RESULTS.append({"benchmark": "outliers_uni_5pct_8sigma", "detector": "grubbs", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- GeneralizedESDDetector -----------------------------------------------
try:
    from dqt.algorithms.outliers_uni.grubbs import GeneralizedESDDetector
    _uni_result("generalized_esd", GeneralizedESDDetector(), threshold=0.01)
except Exception as e:
    print(f"ERROR generalized_esd: {e}")
    RESULTS.append({"benchmark": "outliers_uni_5pct_8sigma", "detector": "generalized_esd", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- DoubleMadOutlierDetector ---------------------------------------------
try:
    from dqt.algorithms.outliers_uni.mad import DoubleMadOutlierDetector
    _uni_result("double_mad_outlier_fraction", DoubleMadOutlierDetector(threshold=6.5))
except Exception as e:
    print(f"ERROR double_mad_outlier_fraction: {e}")
    RESULTS.append({"benchmark": "outliers_uni_5pct_8sigma", "detector": "double_mad_outlier_fraction", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- OutlierFractionRangeDetector (meta-detector) -------------------------
try:
    from dqt.algorithms.outliers_uni.outlier_fraction_range import OutlierFractionRangeDetector
    # history of outlier fractions from a clean baseline (all near 0)
    history = pd.DataFrame({"outlier_fraction": RNG.uniform(0.0, 0.01, 30)})
    current_with_spike = pd.DataFrame({"outlier_fraction": [0.08]})  # spiked
    det = OutlierFractionRangeDetector()
    state  = det.fit(history)
    result = det.score(current_with_spike, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.5)
    RESULTS.append({"benchmark": "outlier_fraction_drift_spike", "detector": "outlier_fraction_drift", "precision": p, "recall": r, "f1": f1})
    print(f"{'outlier_fraction_drift':<35}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR outlier_fraction_drift: {e}")
    RESULTS.append({"benchmark": "outlier_fraction_drift_spike", "detector": "outlier_fraction_drift", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- AutoOutlierDetector --------------------------------------------------
try:
    from dqt.algorithms.outliers_uni.auto_outlier import AutoOutlierDetector
    _uni_result("auto_outlier", AutoOutlierDetector(), threshold=0.01)
except Exception as e:
    print(f"ERROR auto_outlier: {e}")
    RESULTS.append({"benchmark": "outliers_uni_5pct_8sigma", "detector": "auto_outlier", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


## 6. Multivariate outlier detectors

**Benchmark**: ref = 2D Normal(0, I) 500 rows; current = 500 rows with 5 % rows at [10, 10]. Score > 0.02 means outliers detected.

In [ ]:

_multi_ref_arr  = RNG.normal(0, 1, (500, 2))
_multi_curr_arr = np.vstack([RNG.normal(0, 1, (475, 2)),
                              np.full((25, 2), 10.0)])  # 5 % extreme outliers
_multi_ref  = pd.DataFrame(_multi_ref_arr,  columns=["x", "y"])
_multi_curr = pd.DataFrame(_multi_curr_arr, columns=["x", "y"])
_multi_labels = np.array([1])

def _multi_result(slug, det, threshold=0.02):
    state  = det.fit(_multi_ref)
    result = det.score(_multi_curr, state)
    p, r, f1 = pr_f1(_multi_labels, result.score, threshold)
    RESULTS.append({"benchmark": "outliers_multi_5pct_extreme", "detector": slug, "precision": p, "recall": r, "f1": f1})
    print(f"{slug:<35}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")

# ---- IsolationForestDetector ----------------------------------------------
try:
    from dqt.algorithms.outliers_multi.isolation_forest import IsolationForestDetector
    _multi_result("isolation_forest_fraction", IsolationForestDetector())
except Exception as e:
    print(f"ERROR isolation_forest_fraction: {e}")
    RESULTS.append({"benchmark": "outliers_multi_5pct_extreme", "detector": "isolation_forest_fraction", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- LOFDetector ----------------------------------------------------------
try:
    from dqt.algorithms.outliers_multi.lof import LOFDetector
    _multi_result("lof", LOFDetector())
except Exception as e:
    print(f"ERROR lof: {e}")
    RESULTS.append({"benchmark": "outliers_multi_5pct_extreme", "detector": "lof", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- MahalanobisDetector --------------------------------------------------
try:
    from dqt.algorithms.outliers_multi.mahalanobis import MahalanobisDetector
    _multi_result("mahalanobis_distance", MahalanobisDetector())
except Exception as e:
    print(f"ERROR mahalanobis_distance: {e}")
    RESULTS.append({"benchmark": "outliers_multi_5pct_extreme", "detector": "mahalanobis_distance", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- HBOSDetector ---------------------------------------------------------
try:
    from dqt.algorithms.outliers_multi.hbos import HBOSDetector
    _multi_result("hbos", HBOSDetector())
except Exception as e:
    print(f"ERROR hbos: {e}")
    RESULTS.append({"benchmark": "outliers_multi_5pct_extreme", "detector": "hbos", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- ECODDetector ---------------------------------------------------------
try:
    from dqt.algorithms.outliers_multi.ecod import ECODDetector
    _multi_result("ecod", ECODDetector())
except Exception as e:
    print(f"ERROR ecod: {e}")
    RESULTS.append({"benchmark": "outliers_multi_5pct_extreme", "detector": "ecod", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- OneClassSVMDetector --------------------------------------------------
try:
    from dqt.algorithms.outliers_multi.one_class_svm import OneClassSVMDetector
    _multi_result("one_class_svm", OneClassSVMDetector(nu=0.05))
except Exception as e:
    print(f"ERROR one_class_svm: {e}")
    RESULTS.append({"benchmark": "outliers_multi_5pct_extreme", "detector": "one_class_svm", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


## 7. Time series detectors (remaining)

Uses the existing `nab_series()` generator. BOCPD is already in section 1.

In [ ]:

def _ts_bench(slug, det, pattern="level_shift", threshold=0.50):
    vals, labels = nab_series(500, 0.05, pattern, RNG)
    ref_df  = pd.DataFrame({"v": vals[:300]})
    curr_df = pd.DataFrame({"v": vals[300:]})
    curr_labels = labels[300:]
    try:
        state  = det.fit(ref_df)
        result = det.score(curr_df, state)
        p, r, f1 = pr_f1(curr_labels, result.score, threshold)
        RESULTS.append({"benchmark": f"nab_{pattern}", "detector": slug, "precision": p, "recall": r, "f1": f1})
        print(f"{slug:<30}  nab_{pattern:<12}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
    except Exception as e:
        print(f"ERROR {slug} nab_{pattern}: {e}")
        RESULTS.append({"benchmark": f"nab_{pattern}", "detector": slug, "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- STLAnomalyDetector ---------------------------------------------------
try:
    from dqt.algorithms.timeseries.stl import STLAnomalyDetector
    for _pat in ["spike", "level_shift"]:
        _ts_bench("stl_residual_zscore", STLAnomalyDetector(period=7), pattern=_pat, threshold=3.0)
except Exception as e:
    print(f"ERROR stl_residual_zscore: {e}")
    for _pat in ["spike", "level_shift"]:
        RESULTS.append({"benchmark": f"nab_{_pat}", "detector": "stl_residual_zscore", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- CUSUMDetector --------------------------------------------------------
try:
    from dqt.algorithms.timeseries.cusum import CUSUMDetector
    _ts_bench("cusum", CUSUMDetector(), pattern="level_shift", threshold=1.0)
except Exception as e:
    print(f"ERROR cusum: {e}")
    RESULTS.append({"benchmark": "nab_level_shift", "detector": "cusum", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- PageHinkleyDetector --------------------------------------------------
try:
    from dqt.algorithms.timeseries.page_hinkley import PageHinkleyDetector
    _ts_bench("page_hinkley", PageHinkleyDetector(), pattern="level_shift", threshold=0.5)
except Exception as e:
    print(f"ERROR page_hinkley: {e}")
    RESULTS.append({"benchmark": "nab_level_shift", "detector": "page_hinkley", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- HoltWintersDetector --------------------------------------------------
try:
    from dqt.algorithms.timeseries.holt_winters import HoltWintersDetector
    _ts_bench("holt_winters", HoltWintersDetector(period=7), pattern="spike", threshold=0.05)
except Exception as e:
    print(f"ERROR holt_winters: {e}")
    RESULTS.append({"benchmark": "nab_spike", "detector": "holt_winters", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- MatrixProfileDetector ------------------------------------------------
try:
    from dqt.algorithms.timeseries.matrix_profile import MatrixProfileDetector
    _ts_bench("matrix_profile", MatrixProfileDetector(window=7), pattern="spike", threshold=0.05)
except Exception as e:
    print(f"ERROR matrix_profile: {e}")
    RESULTS.append({"benchmark": "nab_spike", "detector": "matrix_profile", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- ProphetAnomalyDetector (optional) ------------------------------------
try:
    from dqt.algorithms.timeseries.prophet_anomaly import ProphetAnomalyDetector
    _ts_bench("prophet_anomaly", ProphetAnomalyDetector(), pattern="spike", threshold=0.05)
except ImportError:
    print("SKIP prophet_anomaly: prophet not installed (install dqt[forecast])")
    RESULTS.append({"benchmark": "nab_spike", "detector": "prophet_anomaly", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})
except Exception as e:
    print(f"ERROR prophet_anomaly: {e}")
    RESULTS.append({"benchmark": "nab_spike", "detector": "prophet_anomaly", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


## 8. Information-theoretic detectors

**Benchmark**: `cramers_v` detects categorical drift; `mutual_information` detects numeric drift.
Ref distribution shifts significantly in current.

In [ ]:

# ---- CramersVDetector -----------------------------------------------------
try:
    from dqt.algorithms.info.cramers_v import CramersVDetector
    _cat_ref_balanced = pd.DataFrame({"v": RNG.choice(["A", "B", "C", "D"], 500)})
    _cat_curr_skewed  = pd.DataFrame({"v": RNG.choice(["A", "B", "C", "D"], 100,
                                                       p=[0.88, 0.04, 0.04, 0.04])})
    det = CramersVDetector()
    state  = det.fit(_cat_ref_balanced)
    result = det.score(_cat_curr_skewed, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.15)
    RESULTS.append({"benchmark": "info_cramers_v_drift", "detector": "cramers_v", "precision": p, "recall": r, "f1": f1})
    print(f"{'cramers_v':<30}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR cramers_v: {e}")
    RESULTS.append({"benchmark": "info_cramers_v_drift", "detector": "cramers_v", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- MutualInformationDetector --------------------------------------------
# NMI is higher = more similar. A large shift → lower NMI → score < threshold.
try:
    from dqt.algorithms.info.mutual_information import MutualInformationDetector
    _nmi_ref  = pd.DataFrame({"v": RNG.normal(0, 1, 500)})
    _nmi_curr = pd.DataFrame({"v": RNG.normal(4, 1, 200)})  # large shift
    det = MutualInformationDetector()
    state  = det.fit(_nmi_ref)
    result = det.score(_nmi_curr, state)
    # Low NMI = drift; flag when NMI < 0.50
    p, r, f1 = pr_f1(np.array([1]), 1.0 - result.score, 0.50)
    RESULTS.append({"benchmark": "info_mutual_info_drift", "detector": "mutual_information", "precision": p, "recall": r, "f1": f1})
    print(f"{'mutual_information':<30}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score(NMI)={result.score:.4f}")
except Exception as e:
    print(f"ERROR mutual_information: {e}")
    RESULTS.append({"benchmark": "info_mutual_info_drift", "detector": "mutual_information", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


## 9. Pattern detector — Benford's Law

**Benchmark**: ref = Benford-conforming data (first digits of 1/x for x in 1..500); dirty = uniform first digits (non-Benford).

In [ ]:

try:
    from dqt.algorithms.pattern.benford import BenfordDetector
    # Benford-conforming: natural numbers produce 1st digits following Benford's law
    _benford_ref  = pd.DataFrame({"v": np.arange(1, 501, dtype=float)})
    # Dirty: force first digits to be uniformly distributed (clearly non-Benford)
    _first_digits_uniform = RNG.integers(1, 10, 500)  # uniform [1,9]
    _benford_dirty = pd.DataFrame({"v": _first_digits_uniform.astype(float)})
    det = BenfordDetector()
    state  = det.fit(_benford_ref)   # fit is a no-op
    result = det.score(_benford_dirty, state)
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.95)
    RESULTS.append({"benchmark": "pattern_benford_uniform", "detector": "benford_law_fit", "precision": p, "recall": r, "f1": f1})
    print(f"{'benford_law_fit':<30}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR benford_law_fit: {e}")
    RESULTS.append({"benchmark": "pattern_benford_uniform", "detector": "benford_law_fit", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


## 10. Referential integrity detector

**Benchmark**: ref lookup set = {1..100}; current FK column has 10 % values outside that set (orphans).

In [ ]:

try:
    from dqt.algorithms.referential.referential import ReferentialIntegrityDetector
    # Pass pre-computed aggregate row: 50 orphans out of 500 rows (10 % violation)
    ref_agg  = _agg_df(orphan_count=0,  total_count=500)
    curr_agg = _agg_df(orphan_count=50, total_count=500)
    det = ReferentialIntegrityDetector(parent_table="dim_products", parent_col="id")
    state  = det.fit(ref_agg)
    result = det.score(curr_agg, state)
    # integrity_rate = 0.90; flag when rate < 1.0
    p, r, f1 = pr_f1(np.array([1]), 1.0 - result.score, 0.05)
    RESULTS.append({"benchmark": "referential_10pct_orphans", "detector": "referential_integrity_rate", "precision": p, "recall": r, "f1": f1})
    print(f"{'referential_integrity_rate':<30}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR referential_integrity_rate: {e}")
    RESULTS.append({"benchmark": "referential_10pct_orphans", "detector": "referential_integrity_rate", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


## 11. Schema change detector

**Benchmark**: ref df has cols [a, b, c]; current has cols [a, b] — column c dropped.

In [ ]:

try:
    from dqt.algorithms.schema.schema_checks import SchemaChangeDetector
    # SchemaChangeDetector expects DataFrames with [col_name, data_type] rows
    ref_schema  = pd.DataFrame({"col_name": ["a", "b", "c"], "data_type": ["int64", "float64", "object"]})
    curr_schema = pd.DataFrame({"col_name": ["a", "b"],      "data_type": ["int64", "float64"]})
    det = SchemaChangeDetector()
    state  = det.fit(ref_schema)
    result = det.score(curr_schema, state)
    # score = 1.0 means change detected
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.5)
    RESULTS.append({"benchmark": "schema_column_dropped", "detector": "schema_change", "precision": p, "recall": r, "f1": f1})
    print(f"{'schema_change':<30}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
    print(f"  detail: {result.plain_english}")
except Exception as e:
    print(f"ERROR schema_change: {e}")
    RESULTS.append({"benchmark": "schema_column_dropped", "detector": "schema_change", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


## 12. Custom detectors

`CallableCheckDetector` wraps any Python callable. `RemoteCheckDetector` requires an HTTP endpoint and is skipped.

In [ ]:

# ---- CallableCheckDetector ------------------------------------------------
try:
    from dqt.algorithms.custom.callable_check import CallableCheckDetector
    # Custom check: fraction of rows where v > 3 (should be ~0 on Normal(0,1))
    def _high_value_frac(df):
        return float((df["v"] > 3).mean())

    ref_clean  = pd.DataFrame({"v": RNG.normal(0, 1, 500)})
    curr_dirty = pd.DataFrame({"v": np.concatenate([RNG.normal(0, 1, 475), np.full(25, 10.0)])})

    det = CallableCheckDetector(fn=_high_value_frac)
    state  = det.fit(ref_clean)
    result = det.score(curr_dirty, state)
    # score = fraction > 3; dirty data has ~5 % at 10.0 so score ≈ 0.05
    p, r, f1 = pr_f1(np.array([1]), result.score, 0.03)
    RESULTS.append({"benchmark": "custom_callable_high_value", "detector": "callable_check", "precision": p, "recall": r, "f1": f1})
    print(f"{'callable_check':<30}  P={p:.3f} R={r:.3f} F1={f1:.3f}  score={result.score:.4f}")
except Exception as e:
    print(f"ERROR callable_check: {e}")
    RESULTS.append({"benchmark": "custom_callable_high_value", "detector": "callable_check", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})

# ---- RemoteCheckDetector (SKIPPED) ----------------------------------------
print("SKIP remote_check: requires a live HTTP endpoint — excluded from offline benchmark")
RESULTS.append({"benchmark": "custom_remote_check", "detector": "remote_check", "precision": float("nan"), "recall": float("nan"), "f1": float("nan")})


## Results

In [ ]:

results_df = pd.DataFrame(RESULTS)

# Summary counts
total   = len(results_df["detector"].unique())
n_nan   = int(results_df.groupby("detector")["f1"].apply(lambda s: s.isna().all()).sum())
n_pass  = total - n_nan
print(f"Detectors attempted : {total}")
print(f"Detectors with results: {n_pass}")
print(f"Detectors with all-NaN (failed/skipped): {n_nan}")
print()

# Full table sorted by detector
full_tbl = (results_df
            .sort_values(["detector", "benchmark"])
            .reset_index(drop=True))
print(full_tbl.to_string(index=False))


In [ ]:

# F1 pivot: mean F1 per (benchmark × detector)
pivot = (results_df
         .pivot_table(index="benchmark", columns="detector", values="f1", aggfunc="mean")
         .round(3))
print("\nF1 by benchmark x detector (NaN = not attempted / skipped):")
print(pivot.to_string())


## Notes

**`warehouse_normal_kpi` MAD F1=0.0 is expected.** `MADOutlierDetector` default threshold (11.0) is calibrated for lognormal revenue data. On Gaussian KPI data the modified Z-score for a 6σ outlier is ~6.0, below the threshold. Use `threshold=3.5` (Iglewicz & Hoaglin) for Gaussian data. See `docs/algorithms/mad_outlier_fraction.md`.

**Aggregate detectors (basic/, referential/, schema/)** receive pre-computed aggregate rows rather than raw data frames. The benchmark directly constructs those rows, which is exactly how the runner delivers data to these detectors after pushing aggregates to the warehouse.

**`prophet_anomaly`** requires the optional `dqt[forecast]` extra (`pip install prophet`). It is skipped gracefully when not installed.

**`remote_check`** requires a live HTTP endpoint and is excluded from this offline benchmark.

**`outlier_fraction_drift`** is a meta-detector: it monitors the time-series of outlier fractions produced by an upstream detector, not raw rows. Its benchmark uses a synthetic history of clean fractions with a spiked current value.